# Chapter 13 Lab — Dialogue Systems

Three chatbots of increasing capability on a small public-domain toy dialogue set (not the
copyrighted movie-script data used in the original course materials): a classic Eliza-style
pattern matcher, a TF-IDF retrieval bot, and a locally-run small instruction-tuned LLM.

## 1. Eliza-style pattern matching (reimplemented from the published algorithm)

In [ ]:
import re, random

PATTERNS = [
    (re.compile(r"I need (.*)", re.I), ["Why do you need {0}?", "Would getting {0} really help you?"]),
    (re.compile(r"I am (.*)", re.I), ["How long have you been {0}?", "Why do you think you are {0}?"]),
    (re.compile(r"(.*)\?$"), ["Why do you ask that?", "What do you think?"]),
    (re.compile(r"(.*)"), ["Tell me more about that.", "I see. Go on."]),
]

def eliza_reply(text):
    for pattern, responses in PATTERNS:
        m = pattern.match(text)
        if m:
            return random.choice(responses).format(*m.groups())
    return "I see."

for line in ["I need a vacation", "I am tired", "Do you understand me?"]:
    print(line, "->", eliza_reply(line))

## 2. TF-IDF retrieval bot

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

candidates = [
    "I can help you book a flight.",
    "Sorry, I didn't understand that.",
    "Your order has been cancelled.",
    "The weather today is sunny with a high of 22 degrees.",
]
vec = TfidfVectorizer().fit(candidates)
cand_vecs = vec.transform(candidates)

def retrieve_reply(user_text):
    v = vec.transform([user_text])
    return candidates[cosine_similarity(v, cand_vecs).argmax()]

print(retrieve_reply("cancel my order please"))
print(retrieve_reply("what's the weather like"))

## 3. A small local generative chatbot

In [ ]:
from transformers import pipeline
chat = pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct")

messages = [{"role": "system", "content": "You are a concise, helpful assistant."},
            {"role": "user", "content": "Can you help me book a flight to Almaty?"}]
out = chat(messages, max_new_tokens=40)
print(out[0]["generated_text"][-1]["content"])

## 4. Side-by-side comparison

In [ ]:
user_input = "cancel my order please"
print("Eliza    :", eliza_reply(user_input))
print("Retrieval:", retrieve_reply(user_input))
gen = chat([{"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": user_input}], max_new_tokens=30)
print("Generative:", gen[0]["generated_text"][-1]["content"])

## Exercise

Send `retrieve_reply` an out-of-scope query ("tell me a joke about penguins"). It will still
confidently return the closest candidate even though none is relevant. Add a similarity-score
threshold below which the bot returns "I'm not sure how to help with that" instead.